In [1]:
import cv2
import time
import numpy as np
from IPython.display import Video
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models.segmentation.deeplabv3 import DeepLabHead

In [2]:
img_root = "C:\\Facultate\\Facultate\\Anul 3\\Semestrul 1\\KBS\\Proiect\\archive\\images"
label_root = "C:\\Facultate\\Facultate\\Anul 3\\Semestrul 1\\KBS\\Proiect\\archive\\labels"

In [3]:
classes_rgb = pd.read_csv(r"C:\\Facultate\\Facultate\\Anul 3\\Semestrul 1\\KBS\\Proiect\\archive\\classes_rgb_values.csv")
video_info = pd.read_csv(r"C:\\Facultate\\Facultate\\Anul 3\\Semestrul 1\\KBS\\Proiect\\archive\\video_info.csv")
rgb_values = np.array(classes_rgb["rgb_values"].apply(eval).tolist())

video_groups = video_info.groupby(["weather", "driving_scenario"])["Index"].apply(list)

In [4]:
# Function - creating the classId matrix
num_classes = len(rgb_values)
unknown_id = 12

def classid(label_rgb, rgb_values):
    h, w, _ = label_rgb.shape
    label_flat = label_rgb.reshape(-1, 3)
    
    class_ids = np.full((label_flat.shape[0],), fill_value=12, dtype=np.int64)

    rgb_values = np.asarray(rgb_values, dtype=np.uint8)

    for idx, rgb in enumerate(rgb_values):
        mask = np.all(label_flat == rgb, axis=1)
        class_ids[mask] = idx

    return class_ids.reshape(h, w)

In [5]:
# Generating a list with all frames
video_names = sorted(os.listdir(img_root))
video_frames = [] # list of (video, (image_path, label_path))

for vid in video_names:
    img_dir = os.path.join(img_root, vid)
    label_dir = os.path.join(label_root, vid)

    frame_names = sorted(os.listdir(img_dir))

    frames = []

    for name in frame_names:
        img_path = os.path.join(img_dir, name)
        label_path = os.path.join(label_dir, name)

        if os.path.isfile(img_path) and os.path.isfile(label_path):
            frames.append((img_path, label_path))
    video_frames.append(frames)

In [6]:
import random
random.seed(42)

train_videos = []
val_videos = []

for (weather, scenario), vids in video_groups.items():
    vids = vids.copy()
    random.shuffle(vids)
    
    n = len(vids)
    
    if n == 1:
        train_videos.append(vids[0])
    else:
        n_val = max(1, int(round(0.3 * n)))
        val_videos.extend(vids[:n_val])
        train_videos.extend(vids[n_val:])

print("Train videos:", len(train_videos), sorted(train_videos))
print("Val videos:", len(val_videos), sorted(val_videos))

Train videos: 20 [0, 2, 3, 5, 6, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 22, 23, 24, 26, 27]
Val videos: 8 [1, 4, 7, 8, 16, 20, 21, 25]


In [7]:
train_frames = []
val_frames = []

for vid in train_videos:
    frames = video_frames[vid]
    train_frames.extend(frames)

for vid in val_videos:
    frames = video_frames[vid]
    val_frames.extend(frames)

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [41]:
class CarlaDataset:
    def __init__(self, frame_list, rgb_values, img_size=None, debug=False, augment=False):
        self.items = frame_list
        self.rgb_values = np.asarray(rgb_values, dtype=np.uint8)
        self.img_size = img_size
        self.debug = debug
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, label_path = self.items[idx]

        # load RGB image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # load RGB label
        label = cv2.imread(label_path)
        label = cv2.cvtColor(label, cv2.COLOR_BGR2RGB)

        if self.img_size is not None:
            w, h = self.img_size
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)
            label = cv2.resize(label, (w, h), interpolation=cv2.INTER_NEAREST)

        
        if self.augment and np.random.rand() < 0.5:
            img = np.ascontiguousarray(img[:, ::-1, :])
            label = np.ascontiguousarray(label[:, ::-1, :])
            
        # convert to class IDs
        class_id_mask = classid(label, self.rgb_values)

        if self.debug and idx < 3:
            unknown_ratio = float((class_id_mask == 12).mean())
            print(f"[DEBUG] idx={idx} unknown_ratio={unknown_ratio:.4f} unique_ids={np.unique(mask)[:30]}")

        img = img.astype(np.float32) / 255.0
        
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        
        img = np.transpose(img, (2, 0, 1))  # CHW format
        
        img_t = torch.from_numpy(img).float()
        mask_t = torch.from_numpy(class_id_mask).long()

        return img_t, mask_t

In [42]:
train_dataset = CarlaDataset(
    frame_list=train_frames,
    rgb_values=rgb_values,
    img_size=(512, 256),
    debug=False,
    augment=True
)
val_dataset = CarlaDataset(
    frame_list=val_frames,
    rgb_values=rgb_values,
    img_size=(512, 256),
    debug=False,
    augment=False
)

In [43]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=2, 
    shuffle=True, 
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,   
    batch_size=2, 
    shuffle=False, 
    num_workers=0
)

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device, print_every=50):
    model.train()
    total_loss = 0.0
    start = time.time()

    for i, (imgs, masks) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(imgs)["out"]
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (i + 1) % print_every == 0:
            elapsed = time.time() - start
            it_s = (i + 1) / elapsed
            print(f"[{i+1}/{len(loader)}] loss={loss.item():.4f} avg_loss={total_loss/(i+1):.4f} it/s={it_s:.2f}")

    return total_loss / len(loader)

In [20]:
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes=13, ignore_index=12, print_every=50):
    model.eval()
    total_loss = 0.0

    inter = torch.zeros(num_classes, dtype=torch.float64)
    union = torch.zeros(num_classes, dtype=torch.float64)

    start = time.time()
    
    for i, (imgs, masks) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        # DeepLab (torchvision) output
        logits = model(imgs)["out"]
        loss = criterion(logits, masks)
        
        total_loss += loss.item()

        preds = logits.argmax(dim=1)  # (B,H,W)

        for cls in range(num_classes):
            if cls == ignore_index:
                continue

            pred_c = (preds == cls)
            mask_c = (masks == cls)

            inter[cls] += (pred_c & mask_c).sum().item()
            union[cls] += (pred_c | mask_c).sum().item()

        if (i + 1) % print_every == 0:
            elapsed = time.time() - start
            it_s = (i + 1) / elapsed
            print(
                f"[VAL {i+1}/{len(loader)}] "
                f"loss={loss.item():.4f} "
                f"avg_loss={total_loss/(i+1):.4f} "
                f"it/s={it_s:.2f}"
            )

    # Compute per-class IoU and mIoU
    per_class_iou = {}
    ious = []

    for cls in range(num_classes):
        if cls == ignore_index:
            continue

        if union[cls] == 0:
            per_class_iou[cls] = None
        else:
            iou = float(inter[cls] / union[cls])
            per_class_iou[cls] = iou
            ious.append(iou)

    miou = sum(ious) / len(ious) if len(ious) > 0 else 0.0
    avg_loss = total_loss / max(1, len(loader))

    return avg_loss, miou, per_class_iou

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = deeplabv3_resnet50(weights="DEFAULT")
model.classifier = DeepLabHead(2048, num_classes)

model = model.to(device)

In [14]:
criterion = nn.CrossEntropyLoss(ignore_index=unknown_id)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

In [18]:
freq = classes_rgb["relative_percentile_frequency"].to_numpy(dtype=np.float32) / 100.0

valid = np.ones_like(freq, dtype=bool)
valid[unknown_id] = False

median_freq = np.median(freq[valid])
weights = np.ones_like(freq, dtype=np.float32)
weights[valid] = median_freq / (freq[valid] + 1e-8)

weights = np.clip(weights, 0.5, 10.0)

weights[unknown_id] = 1.0

class_weights = torch.tensor(weights, device=device, dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=unknown_id)
print("class weights:", weights)

class weights: [10.          0.5         5.142839    4.9655      1.4117633   1.9459434
  0.77419317  0.5         0.5         0.5         0.5         1.7560953
  1.        ]


In [47]:
train_loss = train_one_epoch(
    model=model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    print_every=50
)

val_loss, val_miou, per_class_iou = validate_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    num_classes=num_classes,
    ignore_index=unknown_id,
    print_every=50
)

print(f"\nEpoch done.")
print(f"Train: loss={train_loss:.4f}")
print(f"Val:   loss={val_loss:.4f}  mIoU={val_miou:.4f}")

print("Per-class IoU:")
for cls in sorted(per_class_iou.keys()):
    iou = per_class_iou[cls]
    if iou is None:
        print(f"  class {cls}: N/A")
    else:
        print(f"  class {cls}: {iou:.4f}")

[50/3826] loss=0.1056 avg_loss=0.0921 it/s=2.34
[100/3826] loss=0.1352 avg_loss=0.0946 it/s=2.41
[150/3826] loss=0.1096 avg_loss=0.0941 it/s=2.38
[200/3826] loss=0.0285 avg_loss=0.0919 it/s=2.37
[250/3826] loss=0.1082 avg_loss=0.0903 it/s=2.36
[300/3826] loss=0.0805 avg_loss=0.0896 it/s=2.36
[350/3826] loss=0.0828 avg_loss=0.0893 it/s=2.37
[400/3826] loss=0.0884 avg_loss=0.0888 it/s=2.38
[450/3826] loss=0.0610 avg_loss=0.0885 it/s=2.38
[500/3826] loss=0.1168 avg_loss=0.0886 it/s=2.37
[550/3826] loss=0.1126 avg_loss=0.0885 it/s=2.37
[600/3826] loss=0.0605 avg_loss=0.0882 it/s=2.38
[650/3826] loss=0.0690 avg_loss=0.0882 it/s=2.38
[700/3826] loss=0.0608 avg_loss=0.0881 it/s=2.38
[750/3826] loss=0.0651 avg_loss=0.0880 it/s=2.38
[800/3826] loss=0.1075 avg_loss=0.0878 it/s=2.37
[850/3826] loss=0.0949 avg_loss=0.0878 it/s=2.37
[900/3826] loss=0.1036 avg_loss=0.0878 it/s=2.37
[950/3826] loss=0.0806 avg_loss=0.0881 it/s=2.38
[1000/3826] loss=0.1000 avg_loss=0.0881 it/s=2.37
[1050/3826] loss=0.0

In [17]:
model.train()

images, masks = next(iter(train_loader))
images = images.to(device)
masks  = masks.to(device)

print("images:", images.shape, images.dtype)   # expect [B,3,H,W], float32
print("masks :", masks.shape, masks.dtype)     # expect [B,H,W], int64
print("mask unique ids (sample):", torch.unique(masks)[:30])

out = model(images)["out"]
print("logits:", out.shape, out.dtype)         # expect [B,num_classes,H,W]

loss = criterion(out, masks)
print("loss:", float(loss))

images: torch.Size([2, 3, 256, 512]) torch.float32
masks : torch.Size([2, 256, 512]) torch.int64
mask unique ids (sample): tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12], device='cuda:0')
logits: torch.Size([2, 13, 256, 512]) torch.float32
loss: 2.572023391723633


In [21]:
print("criterion ignore_index:", criterion.ignore_index, "unknown_id:", unknown_id)

criterion ignore_index: 12 unknown_id: 12


In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    verbose=True
)

In [26]:
for param_group in optimizer.param_groups:
    param_group["lr"] = 1e-4
print("New LR:", optimizer.param_groups[0]["lr"])

New LR: 0.0001


In [45]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 6,  # change if needed
    "val_miou": val_miou
}

torch.save(checkpoint, "epoch6chk.pth")
print("Checkpoint saved.")

Checkpoint saved.


In [46]:
criterion = nn.CrossEntropyLoss(ignore_index=unknown_id)  # NO weights

In [ ]:
model = deeplabv3_resnet50(weights=None)
model.classifier = DeepLabHead(2048, num_classes)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

checkpoint = torch.load("deeplab_epoch1.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"]
print("Loaded checkpoint from epoch:", start_epoch)